In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import medfilt
from lmfit.models import LorentzianModel
from utils.plot_spectra import plot_overlay

DATA_DIR = Path("data/spectra_mapping_high_resolution")
OUTPUT_DIR = Path("output/spectra_mapping_high_resolution")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
def load_mapping_txt(filepath: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """マッピングデータ(.txt)を読み込み、(x, y, wavenumber, spectra) を返す"""
    with open(filepath, "r") as f:
        lines = [line.strip().split() for line in f if line.strip()]

    wavenumber = np.array([float(v) for v in lines[0]])
    data = np.array([[float(v) for v in row] for row in lines[1:]])
    x = data[:, 0]
    y = data[:, 1]
    spectra = data[:, 2:]
    return x, y, wavenumber, spectra


def save_spectra_csv(filepath: Path, wavenumber: np.ndarray, spectra: np.ndarray) -> None:
    """全ピクセルの波形を (Wavenumber, Pixel_0, ..., Pixel_N) 形式でCSV保存する"""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    cols = {"Wavenumber": wavenumber}
    for i in range(len(spectra)):
        cols[f"Pixel_{i}"] = spectra[i]
    pd.DataFrame(cols).to_csv(filepath, index=False)


def fit_peak(
    wavenumber: np.ndarray,
    spectrum: np.ndarray,
    half_window: float = 40.0,
) -> tuple[float, float, float, np.ndarray, np.ndarray]:
    """スパイク除去・背景差し引き・ローレンツフィッティングを行い、(center, height, fwhm, processed_spec, fit_curve) を返す"""
    # 1. 宇宙線スパイク除去 (局所中央値から大きく外れるスパイクのみ置換)
    med = medfilt(spectrum, kernel_size=35)
    mad = np.median(np.abs(spectrum - med)) + 1e-6
    clean_spec = np.where((spectrum - med) > 4.0 * mad, med, spectrum)
    smooth_spec = medfilt(clean_spec, kernel_size=5)

    # 2. 両端のCCDロールオフを除外した中央領域からピーク位置の初期値を探索
    search_mask = (wavenumber >= wavenumber.min() + 65.0) & (wavenumber <= wavenumber.max() - 65.0)
    xs, ys = wavenumber[search_mask], smooth_spec[search_mask]
    slope_s = (np.mean(ys[-10:]) - np.mean(ys[:10])) / (np.mean(xs[-10:]) - np.mean(xs[:10]))
    bkg_s = np.mean(ys[:10]) + slope_s * (xs - np.mean(xs[:10]))
    center_init = float(xs[10:-10][np.argmax((ys - bkg_s)[10:-10])])

    # 3. ピーク領域と左端ロールオフを除いた背景領域に3次多項式をフィットして差し引く
    bg_mask = (wavenumber >= wavenumber.min() + 20.0) & (
        (wavenumber < center_init - half_window) | (wavenumber > center_init + half_window)
    )
    baseline = np.polyval(np.polyfit(wavenumber[bg_mask], clean_spec[bg_mask], deg=3), wavenumber)
    processed_spec = clean_spec - baseline

    # 4. ピーク周辺の局所ウィンドウ (center_init ± half_window) でローレンツ関数フィッティング
    fit_mask = (wavenumber >= center_init - half_window) & (wavenumber <= center_init + half_window)
    x_fit = wavenumber[fit_mask]
    y_fit = processed_spec[fit_mask]

    height_init = float(max(np.max((smooth_spec - baseline)[fit_mask]), 1.0))
    sigma_init = 6.0

    model = LorentzianModel()
    params = model.make_params(
        center=center_init,
        amplitude=height_init * np.pi * sigma_init,
        sigma=sigma_init,
    )
    params["center"].set(min=center_init - 15.0, max=center_init + 15.0)
    params["amplitude"].set(min=0.0)
    params["sigma"].set(min=1.5, max=30.0)

    result = model.fit(y_fit, params, x=x_fit)
    center = float(result.params["center"].value)
    height = float(result.params["height"].value)
    fwhm = float(result.params["fwhm"].value)
    fit_curve = model.eval(result.params, x=wavenumber)
    return center, height, fwhm, processed_spec, fit_curve


In [ ]:
for sample_dir in sorted(DATA_DIR.iterdir()):
    if not sample_dir.is_dir():
        continue

    sample_name = sample_dir.name
    sample_out_dir = OUTPUT_DIR / sample_name
    sample_out_dir.mkdir(parents=True, exist_ok=True)

    g_file = next(sample_dir.glob("*_G.txt"))
    twod_file = next(sample_dir.glob("*_2D.txt"))

    x_g, y_g, wn_g, spec_g = load_mapping_txt(g_file)
    x_2d, y_2d, wn_2d, spec_2d = load_mapping_txt(twod_file)

    proc_g_list, fit_g_list = [], []
    proc_2d_list, fit_2d_list = [], []
    rows = []

    for i in range(len(x_g)):
        g_center, g_intensity, g_fwhm, proc_g, fit_g = fit_peak(wn_g, spec_g[i], half_window=35.0)
        twod_center, twod_intensity, twod_fwhm, proc_2d, fit_2d = fit_peak(wn_2d, spec_2d[i], half_window=45.0)

        proc_g_list.append(proc_g)
        fit_g_list.append(fit_g)
        proc_2d_list.append(proc_2d)
        fit_2d_list.append(fit_2d)

        rows.append({
            "X": x_g[i],
            "Y": y_g[i],
            "G_Center": g_center,
            "G_Intensity": g_intensity,
            "G_FWHM": g_fwhm,
            "2D_Center": twod_center,
            "2D_Intensity": twod_intensity,
            "2D_FWHM": twod_fwhm,
            "2D_G_Ratio": twod_intensity / (g_intensity + 1e-12),
        })

    # 1. 処理前波形 (raw/)
    save_spectra_csv(sample_out_dir / "raw" / f"{sample_name}_G_raw.csv", wn_g, spec_g)
    save_spectra_csv(sample_out_dir / "raw" / f"{sample_name}_2D_raw.csv", wn_2d, spec_2d)

    # 2. 処理後波形・フィッティング曲線 (processed/)
    proc_g_arr, fit_g_arr = np.array(proc_g_list), np.array(fit_g_list)
    proc_2d_arr, fit_2d_arr = np.array(proc_2d_list), np.array(fit_2d_list)
    save_spectra_csv(sample_out_dir / "processed" / f"{sample_name}_G_processed.csv", wn_g, proc_g_arr)
    save_spectra_csv(sample_out_dir / "processed" / f"{sample_name}_G_fit.csv", wn_g, fit_g_arr)
    save_spectra_csv(sample_out_dir / "processed" / f"{sample_name}_2D_processed.csv", wn_2d, proc_2d_arr)
    save_spectra_csv(sample_out_dir / "processed" / f"{sample_name}_2D_fit.csv", wn_2d, fit_2d_arr)

    # 3. 波形重ね合わせプロット出力
    plot_overlay(sample_out_dir / f"{sample_name}_G_all_spectra.png", wn_g, spec_g, proc_g_arr, fit_g_arr, title=f"{sample_name} G")
    plot_overlay(sample_out_dir / f"{sample_name}_2D_all_spectra.png", wn_2d, spec_2d, proc_2d_arr, fit_2d_arr, title=f"{sample_name} 2D")

    # 4. フィッティング定量結果CSV
    df = pd.DataFrame(rows)
    out_csv = sample_out_dir / f"{sample_name}_fitting_results.csv"
    df.to_csv(out_csv, index=False)

    print(f"=== {sample_name} ({len(df)} pixels) -> {sample_out_dir} ===")
    print(df[["G_Center", "G_Intensity", "2D_Center", "2D_Intensity", "2D_G_Ratio"]].describe().loc[["mean", "50%", "min", "max"]].round(2))
    print()


=== S64 (36 pixels) -> output/spectra_mapping_high_resolution/S64 ===
      G_Center  G_Intensity  2D_Center  2D_Intensity  2D_G_Ratio
mean   1597.67        36.61    2690.23         19.53        0.54
50%    1598.03        36.04    2689.98         18.77        0.53
min    1590.02        29.14    2688.31         14.89        0.30
max    1599.77        54.33    2693.32         27.62        0.76

=== S72 (36 pixels) -> output/spectra_mapping_high_resolution/S72 ===
      G_Center  G_Intensity  2D_Center  2D_Intensity  2D_G_Ratio
mean   1561.32        13.58    2634.45         11.82        0.99
50%    1561.32        10.40    2634.20         11.41        1.04
min    1555.40         7.66    2627.36          8.21        0.31
max    1566.32        34.09    2641.58         22.16        1.49

=== S73 (36 pixels) -> output/spectra_mapping_high_resolution/S73 ===
      G_Center  G_Intensity  2D_Center  2D_Intensity  2D_G_Ratio
mean   1564.46        16.73    2625.41         12.36        0.92
50%    1